# Databricsk Auto Loader

## What is it
Auto Loader is Databricks **files ingestion tool**<br>

## Sources
It supports following **sources**:
<li>Amazon S3 (s3://)
<li>Azure Data Lake Storage Gen2 (ADLS Gen2, abfss://)
<li>Google Cloud Storage (GCS, gs://)
<li>Azure Blob Storage (wasbs://)<br>

## File formats
It support following **file formats**:
<li>JSON
<li>CSV
<li>XML
<li>PARQUET
<li>AVRO
<li>ORC
<li>TEXT
<li>BINARYFILE<br>

## Docs
It is build around **Spark Structured Streaming** [link to docs](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) <br>
Databricks documentation [here](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/) 

## Syntax
### Using Auto Loader to load to a Unity Catalog managed table
```python
checkpoint_path = "s3://dev-bucket/_checkpoint/dev_table"

(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load("s3://autoloader-source/json-data")
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable("dev_catalog.dev_database.dev_table"))
```
### Configure schema inference and evolution
Specifying a target directory for the option `cloudFiles.schemaLocation` enables **schema inference and evolution**.<br>
Auto Loader samples the first 50 GB or 1000 files that it discovers, whichever limit is crossed first.<br>
Auto Loader stores the schema information in a directory `_schemas`<br>
Auto Loader infers all columns as strings for CSV, JSON, XML. You can force schema inference by `.option("cloudFiles.inferColumnTypes", true)`
AVRO and Parquet inferes schema from files. <br>

```python
(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  # The schema location directory keeps track of your data schema over time
  .option("cloudFiles.schemaLocation", "<path-to-checkpoint>")
  # For CSV files to define header
  .option("header", True)
  # Force schema inference from JSON, CVS and XML files
  .option("cloudFiles.inferColumnTypes", True)
  # Use schema hints
  .option("cloudFiles.schemaHints", "column_name datatype, quantity int, sex string")
  # allow schema evolution
  .option("cloudFiles.schemaEvolutionMode","addNewColumns")
  .load("<path-to-source-data>")
  .writeStream
  .option("checkpointLocation", "<path-to-checkpoint>")
  #
  .option("mergeSchema",True)
  .start("<path_to_target")
)
``` 
### Auto Loader syntax for DLT
If you use Delta Live Tables, Databricks manages schema location and other checkpoint information automatically.<br>
Python
``` python
@dlt.table
def customers():
  return (
    spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .load("/databricks-datasets/retail-org/customers/")
  )

@dlt.table
def sales_orders_raw():
  return (
    spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "json")
      .load("/databricks-datasets/retail-org/sales_orders/")
  )
```

SQL
```sql
CREATE OR REFRESH STREAMING TABLE customers
AS SELECT * FROM read_files("/databricks-datasets/retail-org/customers/", "csv")

CREATE OR REFRESH STREAMING TABLE sales_orders_raw
AS SELECT * FROM read_files("/databricks-datasets/retail-org/sales_orders/", "json")
```

Specify schema manually
Python
```python
@dlt.table
def wiki_raw():
  return (
    spark.readStream.format("cloudFiles")
      .schema("title STRING, id INT, revisionId INT, revisionTimestamp TIMESTAMP, revisionUsername STRING, revisionUsernameId INT, text STRING")
      .option("cloudFiles.format", "parquet")
      .load("/databricks-datasets/wikipedia-datasets/data-001/en_wikipedia/articles-only-parquet")
  )
``` 
SQL
```sql
CREATE OR REFRESH STREAMING TABLE wiki_raw
AS SELECT *
  FROM read_files(
    "/databricks-datasets/wikipedia-datasets/data-001/en_wikipedia/articles-only-parquet",
    "parquet",
    map("schema", "title STRING, id INT, revisionId INT, revisionTimestamp TIMESTAMP, revisionUsername STRING, revisionUsernameId INT, text STRING")
  )
``` 


In [0]:
#variables
path_to_source_data = "abfss://titanic@stdbexvolsandboxjba01.dfs.core.windows.net/landing"
path_to_checkpoint = "abfss://titanic@stdbexvolsandboxjba01.dfs.core.windows.net/landing/_checkpoint"
path_to_target = "titanic.bronze.titanic_raw_autoloader"

# Delete checkpoint if needed to force re-processing
dbutils.fs.rm(path_to_checkpoint, recurse=True)

#autolader
df_titanic_bronze_autoloader = (spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "csv")
  # The schema location directory keeps track of your data schema over time
  .option("cloudFiles.schemaLocation", path_to_checkpoint)
  # Force schema inference from JSON, CVS and XML files
  .option("cloudFiles.inferColumnTypes", True)
  # allow schema evolution
  .option("cloudFiles.schemaEvolutionMode","addNewColumns")
  .option("header", True)  # Ensure CSV header is recognized
  .load(path_to_source_data)
  .writeStream
  .option("checkpointLocation", path_to_checkpoint)
  .option("mergeSchema",True)
  .outputMode("append")  # Ensure records are added
  .toTable(path_to_target)
)


In [0]:
%sql
Select * from titanic.bronze.titanic_raw_autoloader

In [0]:
#Autoloader using DLT

# import libraries
import dlt

@dlt.table(
  table_properties= {"quality": "bronze"},
  comment = "Titanic raw data",
  name = "titanic.bronze.titanic_raw_autoloader_dlt"
)
def titanic_bronze_autoloader_dlt():
  df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "csv") \
        .option("cloudFiles.inferColumnTypes", True) \
        .option("header", True) \
        .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
        .load("abfss://titanic@stdbexvolsandboxjba01.dfs.core.windows.net/landing")
  return df